In [ ]:
import glob
import os
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from tslearn.clustering import TimeSeriesKMeans


# =========================================
# 設定
# =========================================
N_CLUSTERS = 3
PREFERRED_COLUMNS = ["price", "close", "value"]
MIN_LENGTH = 10
RANDOM_STATE = 42

OUTPUT_RESULT_CSV = "clustering_result.csv"
OUTPUT_PLOT_PNG = "clustering_plot.png"


# =========================================
# ログ表示
# =========================================
def log(msg: str):
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{now}] {msg}", flush=True)


# =========================================
# 数値系列を抽出
# =========================================
def extract_numeric_series(df: pd.DataFrame, min_length: int = 10):
    # 優先列を先に探す
    for col in PREFERRED_COLUMNS:
        if col in df.columns:
            s = pd.to_numeric(df[col], errors="coerce")
            s = s.replace([np.inf, -np.inf], np.nan).dropna()
            if len(s) >= min_length:
                return s.to_numpy(dtype=float), col

    # 優先列がなければ、使える数値列を全部調べる
    candidates = []
    for col in df.columns:
        s = pd.to_numeric(df[col], errors="coerce")
        s = s.replace([np.inf, -np.inf], np.nan).dropna()
        if len(s) >= min_length:
            candidates.append((col, s))

    if not candidates:
        return None, None

    # 一番長い列を採用
    best_col, best_series = max(candidates, key=lambda x: len(x[1]))
    return best_series.to_numpy(dtype=float), best_col


# =========================================
# CSV一覧取得
# =========================================
files = sorted(glob.glob("*.csv"))
log(f"検出CSVファイル数: {len(files)}")

if len(files) == 0:
    raise FileNotFoundError("カレントディレクトリにCSVファイルがありません。")

series_list = []
file_names = []
used_columns = []
original_lengths = []

# =========================================
# CSV読み込み
# =========================================
for f in tqdm(files, desc="CSV読み込み"):
    try:
        df = pd.read_csv(f)

        if df.shape[1] == 0:
            log(f"[SKIP] 列がありません: {f}")
            continue

        data, used_col = extract_numeric_series(df, min_length=MIN_LENGTH)

        if data is None:
            log(f"[SKIP] 使用可能な数値列なし: {f}")
            continue

        data = np.asarray(data, dtype=float)
        data = data[np.isfinite(data)]

        if len(data) < MIN_LENGTH:
            log(f"[SKIP] 長さ不足: {f} length={len(data)}")
            continue

        series_list.append(data)
        file_names.append(os.path.basename(f))
        used_columns.append(used_col)
        original_lengths.append(len(data))

        log(f"[OK] {f} -> 使用列={used_col}, 長さ={len(data)}")

    except Exception as e:
        log(f"[ERROR] {f} -> {e}")

usable_count = len(series_list)
log(f"使用可能な時系列数: {usable_count}")

if usable_count == 0:
    raise ValueError("使える時系列データが1件もありません。CSVの列内容を確認してください。")

if usable_count < N_CLUSTERS:
    raise ValueError(
        f"使用可能な時系列数 {usable_count} が、クラスタ数 {N_CLUSTERS} より少ないです。"
    )

# =========================================
# 長さをそろえる
# =========================================
min_len = min(len(s) for s in series_list)
log(f"全系列を最短長 {min_len} にそろえます")

X = np.array([s[:min_len] for s in series_list], dtype=float)

# =========================================
# 系列ごとに標準化
# =========================================
means = X.mean(axis=1, keepdims=True)
stds = X.std(axis=1, keepdims=True)
stds[stds == 0] = 1.0
X_scaled = (X - means) / stds

# tslearn 用に (N, T, 1) へ変形
X_ts = X_scaled[:, :, np.newaxis]

# =========================================
# クラスタリング
# =========================================
log("DTWクラスタリング開始")

model = TimeSeriesKMeans(
    n_clusters=N_CLUSTERS,
    metric="dtw",
    verbose=True,
    random_state=RANDOM_STATE
)

labels = model.fit_predict(X_ts)

log("DTWクラスタリング完了")

# =========================================
# 結果表作成
# =========================================
result_df = pd.DataFrame({
    "file_name": file_names,
    "used_column": used_columns,
    "original_length": original_lengths,
    "aligned_length": [min_len] * len(file_names),
    "cluster": labels
})

result_df = result_df.sort_values(["cluster", "file_name"]).reset_index(drop=True)
result_df.to_csv(OUTPUT_RESULT_CSV, index=False, encoding="utf-8-sig")
log(f"結果CSV保存: {OUTPUT_RESULT_CSV}")

# =========================================
# クラスタごとの内容を表示
# =========================================
for c in range(N_CLUSTERS):
    members = result_df[result_df["cluster"] == c]
    log("")
    log(f"=== クラスタ {c} ===")
    log(f"件数: {len(members)}")
    for _, row in members.iterrows():
        print(
            f"  {row['file_name']}  "
            f"(列={row['used_column']}, 元長さ={row['original_length']})"
        )

# =========================================
# 可視化
# =========================================
fig, axes = plt.subplots(N_CLUSTERS, 1, figsize=(14, 4 * N_CLUSTERS), squeeze=False)
axes = axes.ravel()

for c in range(N_CLUSTERS):
    ax = axes[c]
    idxs = np.where(labels == c)[0]

    for idx in idxs:
        ax.plot(X_scaled[idx], alpha=0.25)

    if len(idxs) > 0:
        centroid = model.cluster_centers_[c].ravel()
        ax.plot(centroid, linewidth=2)

    ax.set_title(f"Cluster {c}  count={len(idxs)}")
    ax.set_xlabel("time step")
    ax.set_ylabel("scaled value")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_PLOT_PNG, dpi=150, bbox_inches="tight")
log(f"プロット保存: {OUTPUT_PLOT_PNG}")
plt.show()

log("すべて完了")

CSVファイル数: 101


読み込み中: 100%|██████████| 101/101 [00:10<00:00, 10.06it/s]


使用データ数: 94
統一長さ: 10
クラスタリング開始...


ValueError: could not convert string to float: 'diemlibre'